In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer

import seaborn as sns 

In [4]:
DATA_DIR = Path.cwd()
DATA_DIR.mkdir(parents=True,exist_ok=True)

In [5]:
DATA_DIR.resolve()

WindowsPath('E:/Data SCience')

In [6]:
customer_profile_raw = pd.read_csv(DATA_DIR / "customer_profile_raw.csv")
transaction_history_raw = pd.read_csv(DATA_DIR / "transaction_history_raw.csv")

In [23]:
display(transaction_history_raw.shape)
customer_profile_raw.shape

(50300, 35)

(25250, 47)

In [24]:
pd.set_option("display.max.Columns",5)

In [26]:
display(customer_profile_raw.head(5))
pd.set_option("display.max.Columns",50)
display(customer_profile_raw.head(5))
print(transaction_history_raw.head(5))

,customer_id,customer_name,email,phone,registration_date,last_purchase_date,birth_date,age,country,state,city,postal_code,preferred_language,gender,marital_status,education_level,occupation,household_size,annual_income,employment_years,membership_tier,...,average_items_per_order,returns_count,complaints_count,website_visits_month,app_sessions_month,email_open_rate,discount_sensitivity,preferred_channel,preferred_category,preferred_payment_method,marketing_opt_in,mobile_app_user,subscription_status,credit_risk_band,customer_satisfaction_score,delivery_satisfaction_score,support_satisfaction_score,engagement_score,churn_risk_score,will_purchase,next_30_day_spend
0,C000001,Customer 000001,customer000001@example.com,15386840849,2017-08-20 00:00:00,2018-07-28,2003-03-27,23,United States,California,Los Angeles,67128,English,Male,Single,Master,Student,2,NaN,2.3,gold,...,2.3,1,0,5,5,0.160,Medium,Website,Home,Bank Transfer,True,True,Inactive,Low,4.0,5,3,48.8,100.0,0.0,0.00
1,C000002,Customer 000002,customer000002@example.com,17397609901,2019-05-17 00:00:00,2026-10-01,1960-10-23,65,India,Gujarat,Surat,16136,Hindi,Male,Married,Master,Professional,3,48950.0,25.5,Bronze,...,2.6,0,0,9,1,0.523,Medium,Marketplace,Grocery,Card,True,True,Inactive,Low,NaN,3,5,60.3,51.4,1.0,58.77
2,C000003,Customer 000003,customer000003@example.com,17344236148,2022-07-23 00:00:00,2026-10-01,1968-03-17,58,United States,California,NaN,57910,French,Male,Married,Bachelor,Self-employed,3,31473.0,21.2,Bronze,...,2.5,1,0,10,7,0.151,Medium,Mobile App,Electronics,Card,True,False,Inactive,Medium,3.0,4,1,65.5,37.8,1.0,71.96
3,C000004,Customer 000004,customer000004@example.com,15337557203,2025-11-15 00:00:00,2026-10-01,1981-07-28,45,India,Gujarat,Ahmedabad,29028,French,Female,Married,Bachelor,Retired,4,NaN,12.4,Bronze,...,3.3,0,1,9,2,0.440,Low,Website,Beauty,Cash,True,False,Active,Low,4.0,4,4,64.2,35.6,1.0,104.32
4,C000005,Customer 000005,customer000005@example.com,13909400296,2025-02-26 00:00:00,2026-10-01,1981-12-07,44,India,Karnataka,Mysuru,28723,Other,Male,Married,Master,Student,3,41461.0,17.3,NaN,...,2.4,0,1,7,6,0.480,Low,Website,Grocery,Wallet,False,False,Active,Medium,4.0,4,4,47.9,35.9,1.0,208.51


,customer_id,customer_name,email,phone,registration_date,last_purchase_date,birth_date,age,country,state,city,postal_code,preferred_language,gender,marital_status,education_level,occupation,household_size,annual_income,employment_years,membership_tier,loyalty_points,total_orders,days_since_last_purchase,average_order_value,total_lifetime_spend,average_items_per_order,returns_count,complaints_count,website_visits_month,app_sessions_month,email_open_rate,discount_sensitivity,preferred_channel,preferred_category,preferred_payment_method,marketing_opt_in,mobile_app_user,subscription_status,credit_risk_band,customer_satisfaction_score,delivery_satisfaction_score,support_satisfaction_score,engagement_score,churn_risk_score,will_purchase,next_30_day_spend
0,C000001,Customer 000001,customer000001@example.com,15386840849,2017-08-20 00:00:00,2018-07-28,2003-03-27,23,United States,California,Los Angeles,67128,English,Male,Single,Master,Student,2,NaN,2.3,gold,440,17,2987,17.55,230.88,2.3,1,0,5,5,0.160,Medium,Website,Home,Bank Transfer,True,True,Inactive,Low,4.0,5,3,48.8,100.0,0.0,0.00
1,C000002,Customer 000002,customer000002@example.com,17397609901,2019-05-17 00:00:00,2026-10-01,1960-10-23,65,India,Gujarat,Surat,16136,Hindi,Male,Married,Master,Professional,3,48950.0,25.5,Bronze,283,6,0,75.72,422.54,2.6,0,0,9,1,0.523,Medium,Marketplace,Grocery,Card,True,True,Inactive,Low,NaN,3,5,60.3,51.4,1.0,58.77
2,C000003,Customer 000003,customer000003@example.com,17344236148,2022-07-23 00:00:00,2026-10-01,1968-03-17,58,United States,California,NaN,57910,French,Male,Married,Bachelor,Self-employed,3,31473.0,21.2,Bronze,301,10,0,45.02,443.42,2.5,1,0,10,7,0.151,Medium,Mobile App,Electronics,Card,True,False,Inactive,Medium,3.0,4,1,65.5,37.8,1.0,71.96
3,C000004,Customer 000004,customer000004@example.com,15337557203,2025-11-15 00:00:00,2026-10-01,1981-07-28,45,India,Gujarat,Ahmedabad,29028,French,Female,Married,Bachelor,Retired,4,NaN,12.4,Bronze,not available,4,0,111.35,398.76,3.3,0,1,9,2,0.440,Low,Website,Beauty,Cash,True,False,Active,Low,4.0,4,4,64.2,35.6,1.0,104.32
4,C000005,Customer 000005,customer000005@example.com,13909400296,2025-02-26 00:00:00,2026-10-01,1981-12-07,44,India,Karnataka,Mysuru,28723,Other,Male,Married,Master,Student,3,41461.0,17.3,NaN,275,3,0,96.92,260.49,2.4,0,1,7,6,0.480,Low,Website,Grocery,Wallet,False,False,Active,Medium,4.0,4,4,47.9,35.9,1.0,208.51


  transaction_id customer_id  transaction_date  order_id product_id  \
0       T0000001     C023588  29-05-2026 00:00  O0000001      P0047   
1       T0000002     C007537  05-08-2017 00:00  O0000002      P0052   
2       T0000003     C001026  19-05-2023 00:00  O0000003      P0043   
3       T0000004     C013693  13-09-2026 00:00  O0000004      P0037   
4       T0000005     C021948  13-01-2024 00:00  O0000005      P0052   

  product_name product_category product_subcategory      brand store_id  \
0  Product 047          Grocery             Premium    UrbanCo     S094   
1  Product 052           Beauty            Seasonal    UrbanCo     S006   
2  Product 043           Sports            Seasonal  Evergreen     S041   
3  Product 037           Sports            Standard  Northstar     S070   
4  Product 052           Beauty            Seasonal    UrbanCo     S100   

  store_type sales_channel         country       city quantity unit_price  \
0     Outlet    Mobile App  United Kingdom   

In [29]:
pd.set_option("display.width",100)
customer_profile_raw

,customer_id,customer_name,email,phone,registration_date,last_purchase_date,birth_date,age,country,state,city,postal_code,preferred_language,gender,marital_status,education_level,occupation,household_size,annual_income,employment_years,membership_tier,loyalty_points,total_orders,days_since_last_purchase,average_order_value,total_lifetime_spend,average_items_per_order,returns_count,complaints_count,website_visits_month,app_sessions_month,email_open_rate,discount_sensitivity,preferred_channel,preferred_category,preferred_payment_method,marketing_opt_in,mobile_app_user,subscription_status,credit_risk_band,customer_satisfaction_score,delivery_satisfaction_score,support_satisfaction_score,engagement_score,churn_risk_score,will_purchase,next_30_day_spend
0,C000001,Customer 000001,customer000001@example.com,15386840849,2017-08-20 00:00:00,2018-07-28,2003-03-27,23,United States,California,Los Angeles,67128,English,Male,Single,Master,Student,2,NaN,2.3,gold,440,17,2987,17.55,230.88,2.3,1,0,5,5,0.160,Medium,Website,Home,Bank Transfer,True,True,Inactive,Low,4.0,5,3,48.8,100.0,0.0,0.00
1,C000002,Customer 000002,customer000002@example.com,17397609901,2019-05-17 00:00:00,2026-10-01,1960-10-23,65,India,Gujarat,Surat,16136,Hindi,Male,Married,Master,Professional,3,48950.0,25.5,Bronze,283,6,0,75.72,422.54,2.6,0,0,9,1,0.523,Medium,Marketplace,Grocery,Card,True,True,Inactive,Low,NaN,3,5,60.3,51.4,1.0,58.77
2,C000003,Customer 000003,customer000003@example.com,17344236148,2022-07-23 00:00:00,2026-10-01,1968-03-17,58,United States,California,NaN,57910,French,Male,Married,Bachelor,Self-employed,3,31473.0,21.2,Bronze,301,10,0,45.02,443.42,2.5,1,0,10,7,0.151,Medium,Mobile App,Electronics,Card,True,False,Inactive,Medium,3.0,4,1,65.5,37.8,1.0,71.96
3,C000004,Customer 000004,customer000004@example.com,15337557203,2025-11-15 00:00:00,2026-10-01,1981-07-28,45,India,Gujarat,Ahmedabad,29028,French,Female,Married,Bachelor,Retired,4,NaN,12.4,Bronze,not available,4,0,111.35,398.76,3.3,0,1,9,2,0.440,Low,Website,Beauty,Cash,True,False,Active,Low,4.0,4,4,64.2,35.6,1.0,104.32
4,C000005,Customer 000005,customer000005@example.com,13909400296,2025-02-26 00:00:00,2026-10-01,1981-12-07,44,India,Karnataka,Mysuru,28723,Other,Male,Married,Master,Student,3,41461.0,17.3,NaN,275,3,0,96.92,260.49,2.4,0,1,7,6,0.480,Low,Website,Grocery,Wallet,False,False,Active,Medium,4.0,4,4,47.9,35.9,1.0,208.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25245,C018162,Customer 018162,customer018162@example.com,14083202419,2022-03-30 00:00:00,2023-04-19,1979-11-06,46,United Kingdom,Scotland,Edinburgh,66759,Hindi,Female,Married,Master,Service,1,48210.0,14.4,Bronze,288,6,1261,69.22,493.57,2.3,1,0,6,5,0.303,Medium,Store,Sports,Cash,False,False,Trial,Medium,4.0,2,5,62.4,68.1,0.0,0.00
25246,C001458,Customer 001458,customer001458@example.com,18154033644,2024-11-20 00:00:00,2026-10-01,1980-07-25,46,Canada,Alberta,Edmonton,65293,Other,Male,Single,High School,Retired,3,NaN,6.2,Bronze,261,7,0,NaN,558.17,3.0,1,0,2,2,0.539,Medium,Marketplace,Home,Cash,True,False,Inactive,Medium,3.0,3,4,33.1,42.1,0.0,0.00
25247,C004557,Customer 004557,customer004557@example.com,18606869008,2023-11-18 00:00:00,2026-10-01,1999-02-11,27,Australia,Victoria,Geelong,25617,Other,Male,Married,Bachelor,Student,3,78746.0,8.1,Bronze,255,4,0,25.91,117.07,2.7,0,0,10,4,0.198,High,Marketplace,Electronics,Bank Transfer,False,True,Active,Low,3.0,3,1,57.2,18.3,1.0,5.31
25248,C007517,Customer 007517,customer007517@example.com,17908303298,2021-07-16 00:00:00,2026-10-01,2007-12-24,18,United States,Texas,Dallas,30193,Spanish,Male,Single,Bachelor,Service,4,40404.0,0.0,Gold,457,7,0,19.39,101.62,1.5,2,1,9,5,0.293,Medium,Mobile App,Beauty,Card,True,True,Inactive,Medium,3.0,3,3,63.4,24.1,0.0,0.00


In [30]:
pd.set_option("display.width",200)
customer_profile_raw

,customer_id,customer_name,email,phone,registration_date,last_purchase_date,birth_date,age,country,state,city,postal_code,preferred_language,gender,marital_status,education_level,occupation,household_size,annual_income,employment_years,membership_tier,loyalty_points,total_orders,days_since_last_purchase,average_order_value,total_lifetime_spend,average_items_per_order,returns_count,complaints_count,website_visits_month,app_sessions_month,email_open_rate,discount_sensitivity,preferred_channel,preferred_category,preferred_payment_method,marketing_opt_in,mobile_app_user,subscription_status,credit_risk_band,customer_satisfaction_score,delivery_satisfaction_score,support_satisfaction_score,engagement_score,churn_risk_score,will_purchase,next_30_day_spend
0,C000001,Customer 000001,customer000001@example.com,15386840849,2017-08-20 00:00:00,2018-07-28,2003-03-27,23,United States,California,Los Angeles,67128,English,Male,Single,Master,Student,2,NaN,2.3,gold,440,17,2987,17.55,230.88,2.3,1,0,5,5,0.160,Medium,Website,Home,Bank Transfer,True,True,Inactive,Low,4.0,5,3,48.8,100.0,0.0,0.00
1,C000002,Customer 000002,customer000002@example.com,17397609901,2019-05-17 00:00:00,2026-10-01,1960-10-23,65,India,Gujarat,Surat,16136,Hindi,Male,Married,Master,Professional,3,48950.0,25.5,Bronze,283,6,0,75.72,422.54,2.6,0,0,9,1,0.523,Medium,Marketplace,Grocery,Card,True,True,Inactive,Low,NaN,3,5,60.3,51.4,1.0,58.77
2,C000003,Customer 000003,customer000003@example.com,17344236148,2022-07-23 00:00:00,2026-10-01,1968-03-17,58,United States,California,NaN,57910,French,Male,Married,Bachelor,Self-employed,3,31473.0,21.2,Bronze,301,10,0,45.02,443.42,2.5,1,0,10,7,0.151,Medium,Mobile App,Electronics,Card,True,False,Inactive,Medium,3.0,4,1,65.5,37.8,1.0,71.96
3,C000004,Customer 000004,customer000004@example.com,15337557203,2025-11-15 00:00:00,2026-10-01,1981-07-28,45,India,Gujarat,Ahmedabad,29028,French,Female,Married,Bachelor,Retired,4,NaN,12.4,Bronze,not available,4,0,111.35,398.76,3.3,0,1,9,2,0.440,Low,Website,Beauty,Cash,True,False,Active,Low,4.0,4,4,64.2,35.6,1.0,104.32
4,C000005,Customer 000005,customer000005@example.com,13909400296,2025-02-26 00:00:00,2026-10-01,1981-12-07,44,India,Karnataka,Mysuru,28723,Other,Male,Married,Master,Student,3,41461.0,17.3,NaN,275,3,0,96.92,260.49,2.4,0,1,7,6,0.480,Low,Website,Grocery,Wallet,False,False,Active,Medium,4.0,4,4,47.9,35.9,1.0,208.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25245,C018162,Customer 018162,customer018162@example.com,14083202419,2022-03-30 00:00:00,2023-04-19,1979-11-06,46,United Kingdom,Scotland,Edinburgh,66759,Hindi,Female,Married,Master,Service,1,48210.0,14.4,Bronze,288,6,1261,69.22,493.57,2.3,1,0,6,5,0.303,Medium,Store,Sports,Cash,False,False,Trial,Medium,4.0,2,5,62.4,68.1,0.0,0.00
25246,C001458,Customer 001458,customer001458@example.com,18154033644,2024-11-20 00:00:00,2026-10-01,1980-07-25,46,Canada,Alberta,Edmonton,65293,Other,Male,Single,High School,Retired,3,NaN,6.2,Bronze,261,7,0,NaN,558.17,3.0,1,0,2,2,0.539,Medium,Marketplace,Home,Cash,True,False,Inactive,Medium,3.0,3,4,33.1,42.1,0.0,0.00
25247,C004557,Customer 004557,customer004557@example.com,18606869008,2023-11-18 00:00:00,2026-10-01,1999-02-11,27,Australia,Victoria,Geelong,25617,Other,Male,Married,Bachelor,Student,3,78746.0,8.1,Bronze,255,4,0,25.91,117.07,2.7,0,0,10,4,0.198,High,Marketplace,Electronics,Bank Transfer,False,True,Active,Low,3.0,3,1,57.2,18.3,1.0,5.31
25248,C007517,Customer 007517,customer007517@example.com,17908303298,2021-07-16 00:00:00,2026-10-01,2007-12-24,18,United States,Texas,Dallas,30193,Spanish,Male,Single,Bachelor,Service,4,40404.0,0.0,Gold,457,7,0,19.39,101.62,1.5,2,1,9,5,0.293,Medium,Mobile App,Beauty,Card,True,True,Inactive,Medium,3.0,3,3,63.4,24.1,0.0,0.00


In [ ]:
sns.set_theme(style="whitegrid")
customer_profile_raw.head(5)

,customer_id,customer_name,email,phone,registration_date,last_purchase_date,birth_date,age,country,state,city,postal_code,preferred_language,gender,marital_status,education_level,occupation,household_size,annual_income,employment_years,membership_tier,loyalty_points,total_orders,days_since_last_purchase,average_order_value,total_lifetime_spend,average_items_per_order,returns_count,complaints_count,website_visits_month,app_sessions_month,email_open_rate,discount_sensitivity,preferred_channel,preferred_category,preferred_payment_method,marketing_opt_in,mobile_app_user,subscription_status,credit_risk_band,customer_satisfaction_score,delivery_satisfaction_score,support_satisfaction_score,engagement_score,churn_risk_score,will_purchase,next_30_day_spend
0,C000001,Customer 000001,customer000001@example.com,15386840849,2017-08-20 00:00:00,2018-07-28,2003-03-27,23,United States,California,Los Angeles,67128,English,Male,Single,Master,Student,2,NaN,2.3,gold,440,17,2987,17.55,230.88,2.3,1,0,5,5,0.160,Medium,Website,Home,Bank Transfer,True,True,Inactive,Low,4.0,5,3,48.8,100.0,0.0,0.00
1,C000002,Customer 000002,customer000002@example.com,17397609901,2019-05-17 00:00:00,2026-10-01,1960-10-23,65,India,Gujarat,Surat,16136,Hindi,Male,Married,Master,Professional,3,48950.0,25.5,Bronze,283,6,0,75.72,422.54,2.6,0,0,9,1,0.523,Medium,Marketplace,Grocery,Card,True,True,Inactive,Low,NaN,3,5,60.3,51.4,1.0,58.77
2,C000003,Customer 000003,customer000003@example.com,17344236148,2022-07-23 00:00:00,2026-10-01,1968-03-17,58,United States,California,NaN,57910,French,Male,Married,Bachelor,Self-employed,3,31473.0,21.2,Bronze,301,10,0,45.02,443.42,2.5,1,0,10,7,0.151,Medium,Mobile App,Electronics,Card,True,False,Inactive,Medium,3.0,4,1,65.5,37.8,1.0,71.96
3,C000004,Customer 000004,customer000004@example.com,15337557203,2025-11-15 00:00:00,2026-10-01,1981-07-28,45,India,Gujarat,Ahmedabad,29028,French,Female,Married,Bachelor,Retired,4,NaN,12.4,Bronze,not available,4,0,111.35,398.76,3.3,0,1,9,2,0.440,Low,Website,Beauty,Cash,True,False,Active,Low,4.0,4,4,64.2,35.6,1.0,104.32
4,C000005,Customer 000005,customer000005@example.com,13909400296,2025-02-26 00:00:00,2026-10-01,1981-12-07,44,India,Karnataka,Mysuru,28723,Other,Male,Married,Master,Student,3,41461.0,17.3,NaN,275,3,0,96.92,260.49,2.4,0,1,7,6,0.480,Low,Website,Grocery,Wallet,False,False,Active,Medium,4.0,4,4,47.9,35.9,1.0,208.51


In [35]:
display(customer_profile_raw.tail())
transaction_history_raw.tail()

,customer_id,customer_name,email,phone,registration_date,last_purchase_date,birth_date,age,country,state,city,postal_code,preferred_language,gender,marital_status,education_level,occupation,household_size,annual_income,employment_years,membership_tier,loyalty_points,total_orders,days_since_last_purchase,average_order_value,total_lifetime_spend,average_items_per_order,returns_count,complaints_count,website_visits_month,app_sessions_month,email_open_rate,discount_sensitivity,preferred_channel,preferred_category,preferred_payment_method,marketing_opt_in,mobile_app_user,subscription_status,credit_risk_band,customer_satisfaction_score,delivery_satisfaction_score,support_satisfaction_score,engagement_score,churn_risk_score,will_purchase,next_30_day_spend
25245,C018162,Customer 018162,customer018162@example.com,14083202419,2022-03-30 00:00:00,2023-04-19,1979-11-06,46,United Kingdom,Scotland,Edinburgh,66759,Hindi,Female,Married,Master,Service,1,48210.0,14.4,Bronze,288,6,1261,69.22,493.57,2.3,1,0,6,5,0.303,Medium,Store,Sports,Cash,False,False,Trial,Medium,4.0,2,5,62.4,68.1,0.0,0.00
25246,C001458,Customer 001458,customer001458@example.com,18154033644,2024-11-20 00:00:00,2026-10-01,1980-07-25,46,Canada,Alberta,Edmonton,65293,Other,Male,Single,High School,Retired,3,NaN,6.2,Bronze,261,7,0,NaN,558.17,3.0,1,0,2,2,0.539,Medium,Marketplace,Home,Cash,True,False,Inactive,Medium,3.0,3,4,33.1,42.1,0.0,0.00
25247,C004557,Customer 004557,customer004557@example.com,18606869008,2023-11-18 00:00:00,2026-10-01,1999-02-11,27,Australia,Victoria,Geelong,25617,Other,Male,Married,Bachelor,Student,3,78746.0,8.1,Bronze,255,4,0,25.91,117.07,2.7,0,0,10,4,0.198,High,Marketplace,Electronics,Bank Transfer,False,True,Active,Low,3.0,3,1,57.2,18.3,1.0,5.31
25248,C007517,Customer 007517,customer007517@example.com,17908303298,2021-07-16 00:00:00,2026-10-01,2007-12-24,18,United States,Texas,Dallas,30193,Spanish,Male,Single,Bachelor,Service,4,40404.0,0.0,Gold,457,7,0,19.39,101.62,1.5,2,1,9,5,0.293,Medium,Mobile App,Beauty,Card,True,True,Inactive,Medium,3.0,3,3,63.4,24.1,0.0,0.00
25249,C020976,Customer 020976,customer020976@example.com,14049700160,2024-10-16 00:00:00,2026-10-01,2004-06-18,22,India,Maharashtra,Pune,12619,Spanish,Male,Single,Master,Service,5,NaN,2.0,Bronze,263,6,0,54.19,263.55,4.8,2,0,9,8,0.084,High,Marketplace,Beauty,Bank Transfer,True,False,Active,Low,NaN,3,5,64.9,46.1,0.0,0.00


,transaction_id,customer_id,transaction_date,order_id,product_id,product_name,product_category,product_subcategory,brand,store_id,store_type,sales_channel,country,city,quantity,unit_price,gross_amount,discount_percentage,discount_amount,tax_rate,tax_amount,shipping_fee,return_flag,return_quantity,payment_method,payment_status,order_status,coupon_code,loyalty_points_earned,delivery_days,customer_rating,transaction_hour,device_type,promotion_flag,transaction_amount
50295,T0034047,C017993,18-10-2025 00:00,O0034047,P0052,Product 052,Beauty,Seasonal,UrbanCo,S192,Outlet,Mobile App,United States,Miami,4,58.13,232.53,0,0.00,8,18.60,7.99,0,0,Wallet,Paid,Shipped,SAVE10,25,3,4.0,17,Mobile,True,259.13
50296,T0012729,C020921,08-06-2026 00:00,O0012729,P0049,Product 049,NaN,Premium,UrbanCo,S059,Partner,Website,India,Mumbai,4,20.3,81.22,20,16.24,18,11.70,0.00,0,0,Cash,Paid,Completed,WELCOME,7,1,5.0,9,Mobile,True,76.67
50297,T0006926,C012273,23-05-2024 00:00,O0006926,P0037,Product 037,Sports,Standard,Northstar,S101,Flagship,Website,United States,San Diego,4,41.42,165.66,0,0.00,5,8.28,4.99,0,0,Cash,Paid,Returned,NONE,17,10,4.0,10,Mobile,False,178.94
50298,T0027996,C004448,06-07-2026 00:00,O0027996,P0052,Product 052,Beauty,Seasonal,UrbanCo,S150,Online,Store,India,Mysuru,2,51.57,103.14,10,10.31,0,0.00,7.99,0,0,Bank Transfer,Paid,Completed,SAVE10,10,0,5.0,0,Desktop,False,100.81
50299,T0037613,C001485,20-01-2025 00:00,O0037613,P0041,Product 041,Grocery,Premium,Northstar,S054,Flagship,Mobile App,United States,San Diego,2,18.62,37.24,15,5.59,8,2.53,4.99,0,0,Cash,Paid,Completed,NONE,3,0,5.0,21,Mobile,True,39.18


In [39]:
transaction_history_raw.dtypes.rename("dtype").to_frame()

,dtype
transaction_id,str
customer_id,str
transaction_date,str
order_id,str
product_id,str
product_name,str
product_category,str
product_subcategory,str
brand,str
store_id,str


In [43]:
seed = 42


In [45]:
transaction_history_raw.sample(5,random_state=seed)

,transaction_id,customer_id,transaction_date,order_id,product_id,product_name,product_category,product_subcategory,brand,store_id,store_type,sales_channel,country,city,quantity,unit_price,gross_amount,discount_percentage,discount_amount,tax_rate,tax_amount,shipping_fee,return_flag,return_quantity,payment_method,payment_status,order_status,coupon_code,loyalty_points_earned,delivery_days,customer_rating,transaction_hour,device_type,promotion_flag,transaction_amount
41046,T0041047,C017651,15-04-2024 00:00,O0041047,P0016,Product 016,NaN,Seasonal,Brightline,S197,Partner,Website,Australia,Sydney,3,17.36,52.09,10,5.21,10,4.69,4.99,0,0,Cash,Paid,Completed,LOYALTY,5,4,4.0,6,Mobile,False,56.56
10132,T0010133,C012536,21-07-2026 00:00,O0010133,P0056,Product 056,Clothing,Basic,Nova,S175,Flagship,Store,United States,San Diego,5,NaN,107.72,15,16.16,18,16.48,7.99,0,0,Wallet,Paid,Completed,NONE,11,7,2.0,1,Mobile,True,116.03
15188,T0015189,C022590,03-10-2023 00:00,O0015189,P0009,Product 009,Clothing,Standard,Brightline,S213,Online,Store,United Kingdom,Cardiff,2,19.55,39.09,0,0.00,18,7.04,4.99,0,0,Cash,Paid,Completed,LOYALTY,5,1,5.0,0,Mobile,False,51.12
37978,T0037979,C005754,01-05-2017 00:00,O0037979,P0028,Product 028,Home,Premium,Nova,S220,Outlet,Marketplace,India,New Delhi,2,21.02,42.04,15,6.31,8,2.86,4.99,0,0,Cash,Paid,Completed,SEASONAL,4,10,5.0,10,Mobile,False,43.58
37315,T0037316,C020311,05-10-2019 00:00,O0037316,P0045,Product 045,Electronics,Seasonal,Northstar,S227,Online,Website,India,Delhi,1,14.85,14.85,5,0.74,18,2.54,7.99,0,0,Cash,Paid,Shipped,SAVE10,2,7,3.0,19,Desktop,False,24.64


In [50]:
print("Columns : ",transaction_history_raw.columns.to_list())

Columns :  ['transaction_id', 'customer_id', 'transaction_date', 'order_id', 'product_id', 'product_name', 'product_category', 'product_subcategory', 'brand', 'store_id', 'store_type', 'sales_channel', 'country', 'city', 'quantity', 'unit_price', 'gross_amount', 'discount_percentage', 'discount_amount', 'tax_rate', 'tax_amount', 'shipping_fee', 'return_flag', 'return_quantity', 'payment_method', 'payment_status', 'order_status', 'coupon_code', 'loyalty_points_earned', 'delivery_days', 'customer_rating', 'transaction_hour', 'device_type', 'promotion_flag', 'transaction_amount']


In [51]:
transaction_history_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 50300 entries, 0 to 50299
Data columns (total 35 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   transaction_id         50300 non-null  str    
 1   customer_id            50300 non-null  str    
 2   transaction_date       49046 non-null  str    
 3   order_id               50300 non-null  str    
 4   product_id             50300 non-null  str    
 5   product_name           50300 non-null  str    
 6   product_category       47296 non-null  str    
 7   product_subcategory    50300 non-null  str    
 8   brand                  50300 non-null  str    
 9   store_id               50300 non-null  str    
 10  store_type             50300 non-null  str    
 11  sales_channel          50300 non-null  str    
 12  country                50300 non-null  str    
 13  city                   50300 non-null  str    
 14  quantity               50300 non-null  str    
 15  unit_price   

In [54]:
transaction_history_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
gross_amount,50300.0,113.675886,122.700144,3.07,34.51,69.75,143.53,877.42
discount_percentage,50300.0,8.471173,9.227324,0.00,0.00,10.00,15.00,150.00
discount_amount,50300.0,9.416096,16.531437,0.00,0.00,3.26,11.22,203.59
tax_rate,50300.0,10.328032,5.859627,-5.00,5.00,10.00,18.00,18.00
tax_amount,50300.0,10.800254,14.858205,0.00,2.23,5.70,13.09,156.09
shipping_fee,50300.0,3.690475,2.622462,0.00,0.00,4.99,4.99,7.99
return_flag,50300.0,0.059006,0.235638,0.00,0.00,0.00,0.00,1.00
return_quantity,50300.0,0.098330,0.661616,0.00,0.00,0.00,0.00,20.00
loyalty_points_earned,50300.0,11.374036,12.561979,0.00,3.00,7.00,14.00,103.00
customer_rating,48290.0,3.973680,0.803159,1.00,3.00,4.00,5.00,9.00


In [52]:
transaction_history_raw.describe(include="all")

,transaction_id,customer_id,transaction_date,order_id,product_id,product_name,product_category,product_subcategory,brand,store_id,store_type,sales_channel,country,city,quantity,unit_price,gross_amount,discount_percentage,discount_amount,tax_rate,tax_amount,shipping_fee,return_flag,return_quantity,payment_method,payment_status,order_status,coupon_code,loyalty_points_earned,delivery_days,customer_rating,transaction_hour,device_type,promotion_flag,transaction_amount
count,50300,50300,49046,50300,50300,50300,47296,50300,50300,50300,50300,50300,50300,50300,50300,48296,50300.000000,50300.000000,50300.000000,50300.000000,50300.000000,50300.000000,50300.000000,50300.000000,50300,47798,50300,50300,50300.000000,48794,48290.000000,50300.000000,50300,50300,48795.000000
unique,50000,21597,3402,50000,60,60,8,4,5,250,4,5,5,34,12,11122,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,4,4,5,NaN,19,NaN,NaN,3,2,NaN
top,T0000148,C999999,07-09-2026 00:00,O0000148,P0054,Product 054,Sports,Standard,UrbanCo,S177,Outlet,Marketplace,United States,Dallas,1,not available,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,Paid,Completed,SEASONAL,NaN,0,NaN,NaN,Mobile,False,NaN
freq,2,35,79,2,911,911,13479,15897,15092,241,12728,12572,17446,2260,10162,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12697,43335,37765,10212,NaN,3322,NaN,NaN,31313,32601,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,113.675886,8.471173,9.416096,10.328032,10.800254,3.690475,0.059006,0.098330,NaN,NaN,NaN,NaN,11.374036,NaN,3.973680,11.531412,NaN,NaN,118.833396
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.700144,9.227324,16.531437,5.859627,14.858205,2.622462,0.235638,0.661616,NaN,NaN,NaN,NaN,12.561979,NaN,0.803159,6.937247,NaN,NaN,125.642819
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.070000,0.000000,0.000000,-5.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,1.000000,0.000000,NaN,NaN,2.710000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.510000,0.000000,0.000000,5.000000,2.230000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,3.000000,NaN,3.000000,6.000000,NaN,NaN,38.130000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.750000,10.000000,3.260000,10.000000,5.700000,4.990000,0.000000,0.000000,NaN,NaN,NaN,NaN,7.000000,NaN,4.000000,11.000000,NaN,NaN,74.360000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143.530000,15.000000,11.220000,18.000000,13.090000,4.990000,0.000000,0.000000,NaN,NaN,NaN,NaN,14.000000,NaN,5.000000,18.000000,NaN,NaN,148.530000


In [59]:
transaction_history_raw.nunique().rename("Count").to_frame()

,Count
transaction_id,50000
customer_id,21597
transaction_date,3402
order_id,50000
product_id,60
product_name,60
product_category,8
product_subcategory,4
brand,5
store_id,250


In [63]:
transaction_history_raw.nunique().sort_values().dropna().to_frame("Count")

,Count
return_flag,2
promotion_flag,2
device_type,3
product_subcategory,4
payment_method,4
return_quantity,4
order_status,4
store_type,4
shipping_fee,4
payment_status,4


In [68]:
thr = transaction_history_raw.nunique(dropna=True).sort_values().to_frame("Count")
thr

,Count
return_flag,2
promotion_flag,2
device_type,3
product_subcategory,4
payment_method,4
return_quantity,4
order_status,4
store_type,4
shipping_fee,4
payment_status,4


In [69]:
def missingness_report(df):
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percentage": (df.isna().mean() * 100).round(2),
        "unique_count": df.nunique(dropna=True)
    })
    return report.sort_values("missing_percentage", ascending=False)



display(missingness_report(customer_profile_raw))

,dtype,missing_count,missing_percentage,unique_count
annual_income,str,3007,11.91,19152
email,str,2518,9.97,22430
customer_satisfaction_score,float64,2023,8.01,6
average_order_value,str,2012,7.97,8314
membership_tier,str,1748,6.92,6
city,str,1516,6.00,34
preferred_channel,str,1501,5.94,5
country,str,1249,4.95,6
registration_date,str,628,2.49,3615
will_purchase,float64,382,1.51,2


In [79]:
def duplicate_summary(df, key=None):
    result = {
        "rows": len(df),
        "exact_duplicate_rows": int(df.duplicated().sum())
    }
    if key and key in df.columns:
        result["missing_key_rows"] = int(df[key].isna().sum())
        result["duplicate_key_rows_after_first"] = int(df.duplicated(subset=[key]).sum())
    return result
print(duplicate_summary(customer_profile_raw))
display(duplicate_summary(customer_profile_raw,"customer_id"))

{'rows': 25250, 'exact_duplicate_rows': 250}


{'rows': 25250,
 'exact_duplicate_rows': 250,
 'missing_key_rows': 0,
 'duplicate_key_rows_after_first': 250}

In [80]:
customer_profile_raw.columns.to_list()

['customer_id',
 'customer_name',
 'email',
 'phone',
 'registration_date',
 'last_purchase_date',
 'birth_date',
 'age',
 'country',
 'state',
 'city',
 'postal_code',
 'preferred_language',
 'gender',
 'marital_status',
 'education_level',
 'occupation',
 'household_size',
 'annual_income',
 'employment_years',
 'membership_tier',
 'loyalty_points',
 'total_orders',
 'days_since_last_purchase',
 'average_order_value',
 'total_lifetime_spend',
 'average_items_per_order',
 'returns_count',
 'complaints_count',
 'website_visits_month',
 'app_sessions_month',
 'email_open_rate',
 'discount_sensitivity',
 'preferred_channel',
 'preferred_category',
 'preferred_payment_method',
 'marketing_opt_in',
 'mobile_app_user',
 'subscription_status',
 'credit_risk_band',
 'customer_satisfaction_score',
 'delivery_satisfaction_score',
 'support_satisfaction_score',
 'engagement_score',
 'churn_risk_score',
 'will_purchase',
 'next_30_day_spend']

In [81]:
display(missingness_report(transaction_history_raw))

,dtype,missing_count,missing_percentage,unique_count
product_category,str,3004,5.97,8
payment_status,str,2502,4.97,4
customer_rating,float64,2010,4.00,6
unit_price,str,2004,3.98,11122
transaction_amount,float64,1505,2.99,22419
delivery_days,str,1506,2.99,19
transaction_date,str,1254,2.49,3402
customer_id,str,0,0.00,21597
transaction_id,str,0,0.00,50000
product_id,str,0,0.00,60


In [82]:

print("Customer duplicates:", duplicate_summary(customer_profile_raw, "customer_id"))
print("Transaction duplicates:", duplicate_summary(transaction_history_raw, "transaction_id"))

Customer duplicates: {'rows': 25250, 'exact_duplicate_rows': 250, 'missing_key_rows': 0, 'duplicate_key_rows_after_first': 250}
Transaction duplicates: {'rows': 50300, 'exact_duplicate_rows': 300, 'missing_key_rows': 0, 'duplicate_key_rows_after_first': 300}
